<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/03_construction_eda_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การสำรวจรายการสัญญาจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี

หน่วยวิเคราะห์ของ Notebook นี้คือหนึ่งชุด `รหัสโครงการ + เลขประจำตัวนิติบุคคล 13 หลัก + เลขที่สัญญา`


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)


In [ ]:
base_dir = Path('/content/drive/MyDrive/learning/dads/dads5001/project_1_dads5001/dataset/procurement/egp-contract')

data_path = base_dir / 'processed' / 'construction_contract_supplier_records_2569.csv'
output_path = base_dir / 'processed' / 'construction_contract_supplier_study_scope_2569.csv'

project_dir = base_dir.parents[2]
figure_dir = project_dir / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

print(f'Input file: {data_path}')
print(f'Output file: {output_path}')
print(f'Figure directory: {figure_dir}')


## 1. เปิดข้อมูลรายการสัญญา


In [ ]:
contract_supplier_data = pd.read_csv(data_path)

print(f'Shape: {contract_supplier_data.shape}')
display(contract_supplier_data.head())


In [ ]:
# ดาวน์โหลดฟอนต์สำหรับแสดงภาษาไทยในกราฟ
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

fm.fontManager.addfont('thsarabunnew-webfont.ttf')

sns.set_theme(style='whitegrid', font='TH Sarabun New')
plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.unicode_minus': False
})

BLUE = '#5B7FA3'
ORANGE = '#D9822B'
GRAY = '#B8C2CC'
TEXT = '#344054'
GRID = '#E4E7EC'


In [ ]:
project_id_column = 'รหัสโครงการ'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
contract_column = 'เลขที่สัญญา'
contract_value_column = 'วงเงินงบประมาณในสัญญา (บาท)'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'


ข้อมูลแต่ละแถวเป็นรายการสัญญา–ผู้รับจ้างตามชุดคีย์ที่กำหนดไว้ใน Notebook 02 การนับรายการต่อจากนี้จึงนับจำนวนแถวโดยตรง


## 2. ภาพรวมมูลค่าสัญญา


In [ ]:
contract_summary = contract_supplier_data[contract_value_column].describe().to_frame('ค่า')
display(contract_summary)


In [ ]:
common_contract_values = (
    contract_supplier_data[contract_value_column]
    .value_counts()
    .head(10)
    .rename_axis(contract_value_column)
    .reset_index(name='จำนวนรายการสัญญา')
)

display(common_contract_values)


In [ ]:
value_band_order = [
    'ไม่เกิน 100,000',
    '100,001–300,000',
    '300,001–400,000',
    '400,001–500,000',
    'มากกว่า 500,000'
]

contract_supplier_data['ช่วงมูลค่าสัญญา'] = pd.cut(
    contract_supplier_data[contract_value_column],
    bins=[-float('inf'), 100000, 300000, 400000, 500000, float('inf')],
    labels=value_band_order
)

value_band_summary = (
    contract_supplier_data
    .groupby('ช่วงมูลค่าสัญญา', observed=False)
    .agg(
        จำนวนรายการสัญญา=(contract_column, 'size'),
        มูลค่าสัญญารวม=(contract_value_column, 'sum')
    )
    .reset_index()
)

value_band_summary['สัดส่วนจำนวน (%)'] = (
    value_band_summary['จำนวนรายการสัญญา'] /
    value_band_summary['จำนวนรายการสัญญา'].sum() * 100
)
value_band_summary['สัดส่วนมูลค่า (%)'] = (
    value_band_summary['มูลค่าสัญญารวม'] /
    value_band_summary['มูลค่าสัญญารวม'].sum() * 100
)

display(value_band_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(
    value_band_summary['ช่วงมูลค่าสัญญา'],
    value_band_summary['สัดส่วนจำนวน (%)'],
    color=BLUE
)
axes[0].set_title('สัดส่วนจำนวนรายการสัญญาตามช่วงมูลค่า')
axes[0].set_xlabel('ร้อยละของรายการสัญญา')
axes[0].set_ylabel('')

axes[1].barh(
    value_band_summary['ช่วงมูลค่าสัญญา'],
    value_band_summary['สัดส่วนมูลค่า (%)'],
    color=ORANGE
)
axes[1].set_title('สัดส่วนมูลค่าสัญญารวมตามช่วงมูลค่า')
axes[1].set_xlabel('ร้อยละของมูลค่าสัญญารวม')
axes[1].set_ylabel('')

for ax in axes:
    ax.invert_yaxis()
    ax.grid(axis='x', color=GRID)
    ax.grid(axis='y', visible=False)
    sns.despine(ax=ax, left=True, bottom=True)

fig.tight_layout()
plt.show()


In [ ]:
png_path = figure_dir / 'fig03_01_contract_records_and_value_by_band.png'
svg_path = figure_dir / 'fig03_01_contract_records_and_value_by_band.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


ตารางและกราฟแสดงสองมุมของช่วงมูลค่าเดียวกัน ได้แก่ สัดส่วนจำนวนรายการสัญญาและสัดส่วนวงเงินรวม เพื่อไม่ให้ช่วงที่มีรายการมากถูกตีความว่าใช้งบประมาณรวมสูงที่สุดโดยอัตโนมัติ


## 3. รายการสัญญารอบ 500,000 บาท


In [ ]:
threshold_data = contract_supplier_data.loc[
    contract_supplier_data[contract_value_column].between(400000, 550000)
].copy()

threshold_data['ช่วงละ 10,000 บาท'] = pd.cut(
    threshold_data[contract_value_column],
    bins=range(400000, 550001, 10000),
    right=True,
    include_lowest=True
)

threshold_summary = (
    threshold_data['ช่วงละ 10,000 บาท']
    .value_counts(sort=False)
    .rename_axis('ช่วงมูลค่าสัญญา')
    .reset_index(name='จำนวนรายการสัญญา')
)

display(threshold_summary)


In [ ]:
colors = [
    ORANGE if interval.left >= 490000 and interval.right <= 500000
    else GRAY if interval.left >= 500000
    else BLUE
    for interval in threshold_summary['ช่วงมูลค่าสัญญา']
]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(
    threshold_summary['ช่วงมูลค่าสัญญา'].astype(str),
    threshold_summary['จำนวนรายการสัญญา'],
    color=colors
)
ax.set_title('จำนวนรายการสัญญารอบ 500,000 บาท')
ax.set_xlabel('วงเงินงบประมาณในสัญญา (บาท)')
ax.set_ylabel('จำนวนรายการสัญญา')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', color=GRID)
ax.grid(axis='x', visible=False)
sns.despine(ax=ax, left=True, bottom=True)
fig.tight_layout()
plt.show()


In [ ]:
png_path = figure_dir / 'fig03_02_contract_value_around_500k.png'
svg_path = figure_dir / 'fig03_02_contract_value_around_500k.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


สีส้มแสดงช่วง 490,000–500,000 บาท และสีเทาแสดงช่วงที่สูงกว่า 500,000 บาท จึงเห็นการเปลี่ยนแปลงของจำนวนรายการก่อนและหลังเกณฑ์ได้ในภาพเดียว


## 4. วิธีจัดซื้อของรายการสัญญาใกล้ 500,000 บาท


In [ ]:
near_ceiling_data = contract_supplier_data.loc[
    contract_supplier_data[contract_value_column].between(490000, 500000)
].copy()

method_summary = (
    near_ceiling_data
    .groupby(method_column, dropna=False)
    .agg(
        จำนวนรายการสัญญา=(contract_column, 'size'),
        มูลค่าสัญญารวม=(contract_value_column, 'sum')
    )
    .sort_values('จำนวนรายการสัญญา', ascending=False)
    .reset_index()
)

method_summary['สัดส่วนจำนวน (%)'] = (
    method_summary['จำนวนรายการสัญญา'] /
    method_summary['จำนวนรายการสัญญา'].sum() * 100
)

display(method_summary)


In [ ]:
method_chart = method_summary.head(8).sort_values('จำนวนรายการสัญญา')
colors = [ORANGE if method == 'เฉพาะเจาะจง' else BLUE for method in method_chart[method_column]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(method_chart[method_column], method_chart['จำนวนรายการสัญญา'], color=colors)
ax.set_title('วิธีจัดซื้อของรายการสัญญาช่วง 490,000–500,000 บาท')
ax.set_xlabel('จำนวนรายการสัญญา')
ax.set_ylabel('')
ax.grid(axis='x', color=GRID)
ax.grid(axis='y', visible=False)
sns.despine(ax=ax, left=True, bottom=True)
fig.tight_layout()
plt.show()


In [ ]:
png_path = figure_dir / 'fig03_03_procurement_method_near_500k.png'
svg_path = figure_dir / 'fig03_03_procurement_method_near_500k.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


ตารางแสดงทั้งจำนวนรายการ สัดส่วนจำนวน และมูลค่ารวม ส่วนกราฟใช้เปรียบเทียบจำนวนรายการระหว่างวิธีจัดซื้อในช่วงใกล้เพดาน


## 5. ข้อกำหนดเกี่ยวกับวงเงิน 500,000 บาท

- พระราชบัญญัติการจัดซื้อจัดจ้างฯ พ.ศ. 2560 มาตรา 55 กำหนดวิธีจัดซื้อจัดจ้างพัสดุ 3 วิธี ได้แก่ วิธีประกาศเชิญชวนทั่วไป วิธีคัดเลือก และวิธีเฉพาะเจาะจง
- มาตรา 56 วรรคหนึ่ง (2)(ข) อนุญาตให้ใช้วิธีเฉพาะเจาะจงเมื่อวงเงินต่อครั้งไม่เกินวงเงินที่กำหนดในกฎกระทรวง
- กฎกระทรวงกำหนดวงเงินดังกล่าวไว้ไม่เกิน 500,000 บาท

แหล่งอ้างอิง: [พระราชบัญญัติการจัดซื้อจัดจ้างฯ ในราชกิจจานุเบกษา](https://www.ratchakitcha.soc.go.th/) และ [กฎหมาย/ระเบียบด้านการจัดซื้อจัดจ้างของกรมบัญชีกลาง](https://www.cgd.go.th/)

ข้อกำหนดนี้ช่วยอธิบายว่าทำไมจึงต้องแยกรายการวิธีเฉพาะเจาะจงที่มีวงเงินไม่เกิน 500,000 บาทไปตรวจรูปแบบการกระจุกตัวต่อ


## 6. ขอบเขตสำหรับตรวจรูปแบบต่อ


In [ ]:
under_500k_mask = contract_supplier_data[contract_value_column].le(500000)
specific_method_mask = contract_supplier_data[method_column].eq('เฉพาะเจาะจง')
study_scope_mask = under_500k_mask & specific_method_mask

contract_supplier_data['อยู่ในขอบเขตตรวจรูปแบบ'] = study_scope_mask
study_scope_data = contract_supplier_data.loc[study_scope_mask].copy()

scope_summary = pd.DataFrame({
    'ขั้นตอน': [
        'รายการสัญญาจ้างก่อสร้างทั้งหมด',
        'วงเงินไม่เกิน 500,000 บาท',
        'วิธีเฉพาะเจาะจงและวงเงินไม่เกิน 500,000 บาท'
    ],
    'จำนวนรายการสัญญา': [
        len(contract_supplier_data),
        int(under_500k_mask.sum()),
        len(study_scope_data)
    ]
})
scope_summary['สัดส่วนจากทั้งหมด (%)'] = (
    scope_summary['จำนวนรายการสัญญา'] / len(contract_supplier_data) * 100
)

display(scope_summary)


ทุกขั้นตอนนับด้วยหน่วยเดียวกันคือหนึ่งชุดรหัสโครงการ–ผู้รับจ้าง–เลขที่สัญญา แถวสุดท้ายคือขอบเขตข้อมูลที่จะนำไปตรวจ Pattern 1–4 ต่อ


In [ ]:
contract_supplier_data.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Saved: {output_path}')
print(f'Rows in study scope: {len(study_scope_data):,}')


## 7. ไฟล์สำหรับวิเคราะห์ต่อ

`construction_contract_supplier_study_scope_2569.csv`

ไฟล์เก็บรายการสัญญาจ้างก่อสร้างทั้งหมด พร้อมคอลัมน์ `อยู่ในขอบเขตตรวจรูปแบบ` สำหรับเลือกข้อมูลใน Notebook 04
